# Saans - Chest X-ray TB Model v2 (four datasets)

Version 2 of the Saans X-ray training notebook. Same model, much more data.

> **Honest scope note:** Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning is the Phase-1 pilot roadmap item.

**What changed from v1**

| | v1 | v2 |
|---|---|---|
| Images | 800 | ~16,000 |
| Sources | NLM Shenzhen + Montgomery | those two, plus TBX11K and the Qatar/Dhaka set |
| "Negative" class | healthy films only | healthy films **and sick-but-not-TB films** |
| Split | train / validation | train / validation / **untouched test** |
| Duplicate check | none | perceptual hash across all four sources |
| Threshold | left at 0.5 | chosen on validation for a recall target |
| Headline metric | accuracy | AUC + recall on the test set |

The two additions that matter most:

**Sick-but-not-TB films.** In v1 every negative was a healthy chest. A model
trained that way only has to learn "does this lung look abnormal", which is a
much easier - and much less useful - question than "is this abnormality TB".
TBX11K contributes ~5,000 films of people who are ill with something that is not
tuberculosis. That is the hard negative class, and it is what v1 was missing.

**A test set that nothing touches.** In v1 the same 160 images picked the early
stopping point and produced the final numbers, which makes those numbers
optimistic. Here, validation drives training and threshold choice, and the test
set is scored exactly once at the end.

**Before you run:** `Runtime > Change runtime type > T4 GPU`, and have your
Kaggle API token ready (Block 1 explains how). Budget roughly 40 minutes of
downloading and 45 minutes of training.

In [ ]:
# Environment check - run this first.
import sys
import tensorflow as tf

print("Python     ", sys.version.split()[0])
print("TensorFlow ", tf.__version__)
print("Keras      ", tf.keras.__version__)

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU        ", [g.name for g in gpus])
    print("            ", tf.config.experimental.get_device_details(gpus[0]).get("device_name", "?"))
else:
    print("GPU         NONE  ->  Runtime > Change runtime type > T4 GPU.")
    print("             With ~16,000 images, CPU training is not realistic. Stop and switch.")

## BLOCK 1 - Download four datasets

**Two come from NIH NLM, no login needed:**

* Shenzhen (China set) - `ChinaSet_AllFiles.zip`, ~662 films
* Montgomery County - `MontgomeryCountyXRaySet.zip`, ~138 films

**Two come from Kaggle, which needs an API token:**

* TBX11K - `usmanshams/tbx-11`, ~11,200 films
* Qatar/Dhaka TB database - `tawsifurrahman/tuberculosis-tb-chest-xray-dataset`, ~4,200 films

### Getting your Kaggle token (one minute, once)

1. Go to <https://www.kaggle.com/settings> while logged in.
2. Scroll to **API** and create a **legacy API key** - that is the one that
   downloads a `kaggle.json` file. Kaggle's newer key types show you a value on
   screen instead and do not produce the file this notebook looks for.
3. The download is silent - no "save as" prompt. Look in your **Downloads**
   folder for `kaggle.json` (it is tiny, about 70 bytes).
4. In Colab, click the **folder icon** on the left, then the **upload** button, and
   upload `kaggle.json` into `/content` (the folder that opens by default).

That is all - the next cell finds it automatically. If you prefer, you can instead
put `KAGGLE_USERNAME` and `KAGGLE_KEY` into Colab's **Secrets** (the key icon on
the left) and the cell will use those.

Labels come from two conventions, depending on the source: the NLM filename
suffix (`..._0.png` = Normal, `..._1.png` = TB), and folder names everywhere else.

In [ ]:
# BLOCK 1 - DOWNLOAD DATA
# Everything runs inside the Colab VM. Nothing touches your laptop.

import json
import os
import shutil
import subprocess
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

DATA_ROOT = Path("/content/saans_data")
RAW_DIR = DATA_ROOT / "raw"
EXTRACT_DIR = DATA_ROOT / "extracted"
RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

NIH_PAGE = "https://lhncbc.nlm.nih.gov/LHC-publications/downloads/TuberculosisChestXrayImageDataSets.html"

# --- the two NLM sets, fetched straight over https -------------------------
NLM_SETS = {
    "shenzhen": {
        "zip": "ChinaSet_AllFiles.zip",
        "prefix": "CHNCXR",
        "urls": [
            "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/China-CXRSet/ChinaSet_AllFiles.zip",
            "https://openi.nlm.nih.gov/imgs/collections/ChinaSet_AllFiles.zip",
        ],
        "kaggle_fallback": "https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-shenzhen",
    },
    "montgomery": {
        "zip": "MontgomeryCountyXRaySet.zip",
        "prefix": "MCUCXR",
        "urls": [
            "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet.zip",
            "https://openi.nlm.nih.gov/imgs/collections/NLM-MontgomeryCXRSet.zip",
        ],
        "kaggle_fallback": "https://www.kaggle.com/datasets/raddar/tuberculosis-chest-xrays-montgomery",
    },
}

# --- the two Kaggle sets ---------------------------------------------------
KAGGLE_SETS = {
    "tbx11k": "usmanshams/tbx-11",
    "qatar": "tawsifurrahman/tuberculosis-tb-chest-xray-dataset",
}


def extract_local_zips():
    """Unzip anything sitting in raw/ or /content into EXTRACT_DIR."""
    candidates = list(RAW_DIR.glob("*.zip")) + list(Path("/content").glob("*.zip"))
    for zpath in sorted(set(candidates)):
        if not zipfile.is_zipfile(zpath):
            continue
        marker = EXTRACT_DIR / (".done_" + zpath.name)
        if marker.exists():
            continue
        print(f"  extracting {zpath.name} ...")
        with zipfile.ZipFile(zpath) as zf:
            zf.extractall(EXTRACT_DIR / zpath.stem)
        marker.write_text("ok")


def nlm_images(prefix):
    """
    NLM films on disk. The Montgomery zip also ships hand-drawn lung masks under
    ManualMask/ with the SAME filenames as the films - excluding any path with
    'mask' in it keeps those binary masks out of the training set.
    """
    return sorted(
        p
        for p in EXTRACT_DIR.rglob("*")
        if p.suffix.lower() in {".png", ".jpg", ".jpeg"}
        and p.name.upper().startswith(prefix)
        and p.stem.rsplit("_", 1)[-1] in {"0", "1"}
        and "mask" not in str(p).lower()
        and "__MACOSX" not in str(p)
    )


def http_download(url, dest):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (Saans training notebook)"})
    with urllib.request.urlopen(req, timeout=180) as resp, open(dest, "wb") as fh:
        total = int(resp.headers.get("Content-Length") or 0)
        done = 0
        while True:
            chunk = resp.read(1 << 20)
            if not chunk:
                break
            fh.write(chunk)
            done += len(chunk)
            tail = f" / {total / 1e6:.1f} MB" if total else ""
            print(f"\r    {done / 1e6:7.1f} MB{tail}", end="")
    print()
    if not zipfile.is_zipfile(dest):
        dest.unlink(missing_ok=True)
        raise RuntimeError("server returned something that is not a zip (an HTML error page?)")


print("=" * 70)
print("NLM sets")
print("=" * 70)
extract_local_zips()

for key, spec in NLM_SETS.items():
    if nlm_images(spec["prefix"]):
        print(f"  {key:11s} already present: {len(nlm_images(spec['prefix']))} films")
        continue
    print(f"\nDownloading {key} ({spec['zip']}) ...")
    for url in spec["urls"]:
        try:
            print(f"  trying {url}")
            http_download(url, RAW_DIR / spec["zip"])
            break
        except (urllib.error.URLError, urllib.error.HTTPError, RuntimeError, TimeoutError, OSError) as exc:
            print(f"  ! failed: {type(exc).__name__}: {exc}")
    extract_local_zips()
    n = len(nlm_images(spec["prefix"]))
    if n:
        print(f"  {key} ready: {n} films")
    else:
        print(f"  Download failed. Go to {spec['kaggle_fallback']}, download the zip manually,")
        print("  and upload it to Colab using the folder icon on the left, then re-run this cell.")

In [ ]:
# BLOCK 1 (cont.) - the two Kaggle sets.
# Repeated here so this cell runs on its own; it still needs EXTRACT_DIR and
# KAGGLE_SETS from the cell above, so run that one first.
import json
import os
import shutil
import subprocess
from pathlib import Path


def ensure_kaggle_auth():
    """The Kaggle CLI wants ~/.kaggle/kaggle.json. Accept it from secrets or an upload."""
    target = Path.home() / ".kaggle" / "kaggle.json"
    if target.exists():
        return True

    # Colab Secrets (key icon on the left): KAGGLE_USERNAME + KAGGLE_KEY
    try:
        from google.colab import userdata

        user, key = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if user and key:
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_text(json.dumps({"username": user, "key": key}))
            target.chmod(0o600)
            print("  using Kaggle credentials from Colab Secrets")
            return True
    except Exception:
        pass

    # a kaggle.json you uploaded with the folder icon. Searched loosely: a second
    # download lands as "kaggle (1).json", and Colab's uploader drops the file
    # wherever the file browser happens to be pointed.
    candidates = [p for p in Path("/content").rglob("*kaggle*.json") if p.is_file()]
    if candidates:
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(candidates[0], target)
        target.chmod(0o600)
        print(f"  using {candidates[0]}")
        return True
    return False


def kaggle_download(name, slug):
    dest = EXTRACT_DIR / name
    if dest.exists() and any(dest.rglob("*.png")) or (dest.exists() and any(dest.rglob("*.jpg"))):
        print(f"  {name:8s} already present")
        return True

    dest.mkdir(parents=True, exist_ok=True)
    print(f"\nDownloading {name} ({slug}) - several GB, this takes a while ...")
    # Output is NOT captured, so the Kaggle CLI's progress bar reaches the cell.
    # Capturing it makes the cell look frozen for 20 minutes.
    proc = subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(dest), "--unzip", "--force"])
    if proc.returncode != 0:
        print(f"  ! kaggle CLI failed for {slug} (exit {proc.returncode}) - see the output above")
        return False
    n = sum(1 for p in dest.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
    print(f"  {name} ready: {n} image files")
    return True


print("=" * 70)
print("Kaggle sets")
print("=" * 70)

subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)

if not ensure_kaggle_auth():
    print()
    print("No Kaggle credentials found. Do this, then re-run this cell:")
    print("  1. open https://www.kaggle.com/settings")
    print("  2. under API, click 'Create New Token' - kaggle.json downloads")
    print("  3. in Colab, click the folder icon on the left and upload kaggle.json into /content")
    raise SystemExit

for name, slug in KAGGLE_SETS.items():
    kaggle_download(name, slug)

In [ ]:
# BLOCK 1 (cont.) - build one labelled index across all four sources.
import pandas as pd

# Folder names that identify a class. Matched against whole path SEGMENTS, never
# as substrings: the folder 'TBX11K' contains the letters 'tb' and a substring
# match would label every image in the dataset as tuberculosis.
TB_DIRS = {"tb", "tuberculosis", "tuberculose", "active_tb", "activetb", "tb_positive", "abnormal_tb"}
NEG_DIRS = {
    "normal", "normals", "health", "healthy", "non_tb", "nontb", "negative",
    "sick", "sick_non_tb", "no_finding", "nofinding", "other",
}

CLASS_NAMES = ["Normal", "TB"]


def label_from_folders(path):
    """Walk the folders from the deepest upward and take the first class we recognise."""
    for part in reversed(path.parts[:-1]):
        seg = part.strip().lower().replace("-", "_").replace(" ", "_")
        if seg in TB_DIRS:
            return 1
        if seg in NEG_DIRS:
            return 0
    return None


def label_for(path):
    name = path.name.upper()
    if name.startswith(("CHNCXR", "MCUCXR")):        # NLM filename convention
        suffix = path.stem.rsplit("_", 1)[-1]
        return int(suffix) if suffix in {"0", "1"} else None
    return label_from_folders(path)


def source_for(path):
    rel = path.relative_to(EXTRACT_DIR).parts[0].lower()
    name = path.name.upper()
    if name.startswith("CHNCXR"):
        return "shenzhen"
    if name.startswith("MCUCXR"):
        return "montgomery"
    if "tbx" in rel:
        return "tbx11k"
    return "qatar" if "tuberculosis" in rel or "qatar" in rel else rel


rows, unlabelled = [], []
for p in EXTRACT_DIR.rglob("*"):
    if p.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
        continue
    if "mask" in str(p).lower() or "__MACOSX" in str(p):
        continue
    label = label_for(p)
    if label is None:
        unlabelled.append(p)
        continue
    rows.append({"path": str(p), "label": label, "source": source_for(p)})

df = pd.DataFrame(rows).drop_duplicates(subset="path")
# NLM films first, so that if the same image appears in two sources the original
# survives the de-duplication step in Block 2.
order = {"shenzhen": 0, "montgomery": 1, "qatar": 2, "tbx11k": 3}
df = df.sort_values(by="source", key=lambda s: s.map(order).fillna(9)).reset_index(drop=True)

print(f"labelled files:   {len(df)}")
print(f"unlabelled files: {len(unlabelled)}  (skipped - no class folder we recognise)")
print()
print(pd.crosstab(df["source"], df["label"].map({0: "Normal", 1: "TB"}), margins=True))

if unlabelled:
    print("\nExamples of what was skipped (folder -> filename):")
    for p in unlabelled[:8]:
        print("   ", p.parent.relative_to(EXTRACT_DIR), "->", p.name)
    print("If a whole dataset is missing above, its folders are named something this")
    print("notebook does not recognise - add those names to TB_DIRS / NEG_DIRS and re-run.")

if df.empty:
    raise RuntimeError("No labelled images found at all - check the download steps above.")

## BLOCK 2 - Preprocess, de-duplicate, three-way split

Same preprocessing as v1 - RGB, 224x224, `densenet.preprocess_input`, flip /
rotate / zoom on the training split only - with two additions.

**De-duplication.** The Qatar/Dhaka database was assembled partly from the same
NLM sources this notebook downloads directly, so the same film can arrive twice
under different names. If one copy landed in training and the other in test, the
test score would be inflated and the model would look better than it is. Every
image gets a small perceptual hash (16x16, thresholded at its own mean); exact
hash collisions are dropped, keeping the NLM original where there is a choice.

**Three-way split, 70 / 15 / 15.** Training fits the weights. Validation picks
the stopping point and the decision threshold. Test is scored once, at the very
end, and influences nothing.

In [ ]:
# BLOCK 2 - PREPROCESS
import os
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import train_test_split
from tensorflow.keras.applications.densenet import preprocess_input
from tqdm.auto import tqdm

IMG_SIZE = 224          # 320 would resolve fine detail better, at ~2x the training time
BATCH_SIZE = 32
SEED = 42

CACHE_X = DATA_ROOT / f"decoded_{IMG_SIZE}.npy"
CACHE_DF = DATA_ROOT / f"decoded_{IMG_SIZE}_index.csv"

tf.keras.utils.set_random_seed(SEED)


def decode_and_hash(path):
    """One decode per file: gives back the 224 training image AND its perceptual hash."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:                                   # cv2 refused it - try PIL
        with Image.open(path) as im:
            img = np.asarray(im.convert("L"))
    tiny = cv2.resize(img, (16, 16), interpolation=cv2.INTER_AREA)
    digest = np.packbits(tiny > tiny.mean()).tobytes().hex()
    big = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.repeat(big[:, :, None], 3, axis=2), digest


if CACHE_X.exists() and CACHE_DF.exists():
    X = np.load(CACHE_X, mmap_mode="r")
    df = pd.read_csv(CACHE_DF)
    print(f"loaded {len(df)} decoded images from cache")
else:
    n = len(df)
    X_all = np.empty((n, IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)   # preallocated: no np.stack copy
    hashes = [None] * n
    with ThreadPoolExecutor(max_workers=8) as pool:
        stream = pool.map(decode_and_hash, df["path"].tolist())
        for i, (arr, digest) in enumerate(tqdm(stream, total=n, desc="decoding")):
            X_all[i] = arr
            hashes[i] = digest

    df = df.assign(phash=hashes)
    keep = ~df.duplicated(subset="phash", keep="first")     # NLM originals sorted first, so they win
    print(f"\nduplicate films dropped: {int((~keep).sum())}")
    X = np.ascontiguousarray(X_all[keep.to_numpy()])
    df = df[keep].reset_index(drop=True)
    del X_all

    np.save(CACHE_X, X)
    df.to_csv(CACHE_DF, index=False)
    print(f"cached -> {CACHE_X}")

y = df["label"].to_numpy().astype("float32")
print(f"\nX: {X.shape} {X.dtype}   ({X.nbytes / 1e9:.2f} GB in RAM)")
print(f"TB fraction: {y.mean():.3f}")
print()
print(pd.crosstab(df["source"], df["label"].map({0: "Normal", 1: "TB"}), margins=True))

In [ ]:
# BLOCK 2 (cont.) - three-way split and the tf.data pipelines.
idx_all = np.arange(len(y))
idx_train, idx_hold = train_test_split(idx_all, test_size=0.30, random_state=SEED, stratify=y)
idx_val, idx_test = train_test_split(idx_hold, test_size=0.50, random_state=SEED, stratify=y[idx_hold])

X_train, y_train = np.asarray(X[idx_train]), y[idx_train]
X_val, y_val = np.asarray(X[idx_val]), y[idx_val]
X_test, y_test = np.asarray(X[idx_test]), y[idx_test]

df_val = df.iloc[idx_val].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)

for name, arr in (("train", y_train), ("val", y_val), ("test", y_test)):
    print(f"{name:5s}: {len(arr):6d}   TB {int(arr.sum()):5d} / Normal {int((1 - arr).sum()):5d}   ({arr.mean():.1%} TB)")

augment = tf.keras.Sequential(
    [
        tf.keras.layers.RandomFlip("horizontal", seed=SEED),
        tf.keras.layers.RandomRotation(0.06, fill_mode="nearest", seed=SEED),   # ~ +/-21 degrees
        tf.keras.layers.RandomZoom(0.10, fill_mode="nearest", seed=SEED),
    ],
    name="augmentation",
)


def make_dataset(images, labels, training):
    ds = tf.data.Dataset.from_tensor_slices((images, labels))
    if training:
        ds = ds.shuffle(min(len(images), 4096), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE)

    def prep(xb, yb):
        xb = tf.cast(xb, tf.float32)
        if training:
            xb = augment(xb, training=True)
        return preprocess_input(xb), yb

    return ds.map(prep, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val, training=False)      # unshuffled: predictions stay aligned
test_ds = make_dataset(X_test, y_test, training=False)
print()
print("steps per epoch:", len(train_ds))

## BLOCK 3 - Train

Same two-phase transfer learning as v1, with the settings that worked, plus
mixed precision so the T4 uses its tensor cores (roughly half the training time).
The final layer is forced back to float32 - a float16 sigmoid loses precision
exactly where the probabilities matter.

Fewer epochs than v1, because each epoch now covers ~20x more images.

Note on metrics: with roughly 12% of films positive, **accuracy is close to
useless** - always predicting "Normal" would score about 88%. Watch AUC, PR-AUC
and recall instead.

In [ ]:
# BLOCK 3 - TRAIN
# Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning
# is the Phase-1 pilot roadmap item.
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import callbacks, layers, models, optimizers

USE_MIXED_PRECISION = True      # set False if you hit anything strange on a non-T4 GPU
if USE_MIXED_PRECISION and tf.config.list_physical_devices("GPU"):
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("mixed precision:", tf.keras.mixed_precision.global_policy().name)

inputs = layers.Input((IMG_SIZE, IMG_SIZE, 3), name="cxr_input")
base = tf.keras.applications.DenseNet121(include_top=False, weights="imagenet", input_tensor=inputs)
x = layers.GlobalAveragePooling2D(name="gap")(base.output)
x = layers.Dropout(0.3, name="drop")(x)
# dtype float32 on the head: keep the probability itself out of float16
outputs = layers.Dense(1, activation="sigmoid", dtype="float32", name="tb_output")(x)
model = models.Model(inputs, outputs, name="saans_xray_densenet121")

METRICS = [
    "accuracy",
    tf.keras.metrics.AUC(name="auc"),
    tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
    tf.keras.metrics.Precision(name="precision"),
    tf.keras.metrics.Recall(name="recall"),
]

class_weight = dict(enumerate(compute_class_weight("balanced", classes=np.array([0.0, 1.0]), y=y_train)))
print("class weights:", {int(k): round(float(v), 3) for k, v in class_weight.items()})

# ---- Phase 1: frozen backbone, train the head only ------------------------
base.trainable = False
model.compile(optimizer=optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=METRICS)
print(f"trainable params (phase 1): {int(sum(np.prod(w.shape) for w in model.trainable_weights)):,}")

hist1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=6,
    class_weight=class_weight,
    callbacks=[callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=3, restore_best_weights=True)],
    verbose=1,
)

In [ ]:
# BLOCK 3 (cont.) - Phase 2: unfreeze from conv4 and fine-tune at 1e-4.
FINE_TUNE_FROM = "conv4_block1_0_bn"

base.trainable = True
names = [l.name for l in base.layers]
if FINE_TUNE_FROM not in names:
    raise RuntimeError(f"{FINE_TUNE_FROM} not found; conv4 layers start: {[n for n in names if n.startswith('conv4')][:5]}")
cut = names.index(FINE_TUNE_FROM)

for layer in base.layers[:cut]:
    layer.trainable = False
for layer in base.layers[cut:]:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False          # keep the pretrained BN statistics

model.compile(optimizer=optimizers.Adam(1e-4), loss="binary_crossentropy", metrics=METRICS)
print(f"unfrozen from layer {cut}/{len(names)} ({FINE_TUNE_FROM})")
print(f"trainable params (phase 2): {int(sum(np.prod(w.shape) for w in model.trainable_weights)):,}")

hist2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    class_weight=class_weight,
    callbacks=[
        callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=5, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ],
    verbose=1,
)

history = {k: list(hist1.history.get(k, [])) + list(hist2.history.get(k, [])) for k in hist2.history if k in hist1.history}
print(f"\nbest validation AUC:    {max(history['val_auc']):.4f}")
print(f"best validation PR-AUC: {max(history['val_pr_auc']):.4f}")
print("(accuracy is not the headline here - the classes are imbalanced)")

## BLOCK 4 - Choose the threshold on validation

Two steps, deliberately separated.

**Here:** score the validation set and pick the decision threshold that hits the
recall target. Recall is the number that matters - a missed case is a child with
TB sent home, a false alarm is one unnecessary follow-up.

**Next cell:** score the test set at that threshold. Those are the numbers you
report, and the test set has had no influence on anything up to this point.

In [ ]:
# BLOCK 4 - THRESHOLD, chosen on validation only
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
import matplotlib.pyplot as plt

TARGET_RECALL = 0.90        # WHO's triage-test benchmark is >=90% sensitivity

val_prob = model.predict(val_ds, verbose=0).ravel()
y_val_true = y_val.astype(int)

print(f"validation AUC:    {roc_auc_score(y_val_true, val_prob):.4f}")
print(f"validation PR-AUC: {average_precision_score(y_val_true, val_prob):.4f}")
print()

print(f"{'threshold':>9} {'missed TB':>10} {'false alarms':>13} {'recall':>8} {'precision':>10}")
for t in (0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1, 0.05):
    pred = (val_prob >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_val_true, pred).ravel()
    print(f"{t:>9.2f} {fn_:>10} {fp_:>13} {tp_ / (tp_ + fn_):>8.3f} {(tp_ / (tp_ + fp_) if tp_ + fp_ else 0):>10.3f}")

prec_c, rec_c, thr_c = precision_recall_curve(y_val_true, val_prob)
ok = rec_c[:-1] >= TARGET_RECALL
THRESHOLD = float(thr_c[ok][-1]) if ok.any() else 0.5
print(f"\nchosen threshold for recall >= {TARGET_RECALL}: {THRESHOLD:.4f}")

In [ ]:
# BLOCK 4 (cont.) - FINAL numbers, on the untouched test set.
test_prob = model.predict(test_ds, verbose=0).ravel()
y_true = y_test.astype(int)
y_pred = (test_prob >= THRESHOLD).astype(int)

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, test_prob)
pr_auc = average_precision_score(y_true, test_prob)

print("=" * 62)
print(f"TEST SET - {len(y_true)} films, scored once, at threshold {THRESHOLD:.4f}")
print("=" * 62)
print(f"  roc auc  : {auc:.4f}")
print(f"  pr auc   : {pr_auc:.4f}")
print(f"  recall   : {rec:.4f}   <- the one that matters for a screening tool")
print(f"  precision: {prec:.4f}")
print(f"  accuracy : {acc:.4f}   (inflated by class imbalance - do not lead with this)")
print()
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(y_true, y_pred)
tn, fp, fn, tp = cm.ravel()
print(pd.DataFrame(cm, index=[f"true {c}" for c in CLASS_NAMES], columns=[f"pred {c}" for c in CLASS_NAMES]))
print(f"\nMissed TB (false negatives): {fn}    False alarms (false positives): {fp}")
print(f"specificity: {tn / (tn + fp):.4f}   (WHO triage benchmark is >= 0.70)")

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Saans CXR v2 - test set")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# BLOCK 4 (cont.) - per-source breakdown on the test set.
# The four datasets were collected on different equipment and have very different
# TB rates (Shenzhen ~51% TB, TBX11K ~9.5%). A model can score well by recognising
# WHICH DATASET a film came from and playing that base rate, instead of finding
# disease. Even AUC across sources means it learned radiology; strong on one and
# weak on another means it learned scanners, and the headline number is misleading.
rows = []
for src in sorted(df_test["source"].unique()):
    m = (df_test["source"] == src).to_numpy()
    if m.sum() < 20 or len(np.unique(y_true[m])) < 2:
        continue
    rows.append(
        {
            "source": src,
            "films": int(m.sum()),
            "TB": int(y_true[m].sum()),
            "recall": round(float(recall_score(y_true[m], y_pred[m], zero_division=0)), 3),
            "precision": round(float(precision_score(y_true[m], y_pred[m], zero_division=0)), 3),
            "roc_auc": round(float(roc_auc_score(y_true[m], test_prob[m])), 3),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# BLOCK 4 (cont.) - training curves, saved to training_curves.png
n1 = len(hist1.history["loss"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key, title in zip(axes, ["auc", "loss", "pr_auc"], ["ROC AUC", "Loss", "PR AUC"]):
    ax.plot(history[key], label=f"train {key}")
    ax.plot(history["val_" + key], label=f"val {key}")
    ax.axvline(n1 - 0.5, color="grey", ls="--", lw=1)
    ax.text(n1 - 0.4, ax.get_ylim()[0], " fine-tune", fontsize=8, color="grey", va="bottom")
    ax.set_title(title)
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(alpha=0.3)
fig.suptitle("Saans X-ray DenseNet121 v2 - training history (phase 1 | phase 2)")
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()
print("saved: training_curves.png, confusion_matrix.png")

## BLOCK 5 - Save

Saves `saans_xray_densenet121.h5`, reloads it from disk and re-scores the test
set to prove the file is intact, and writes a metadata sidecar carrying the
chosen threshold and the test metrics.

In [ ]:
# BLOCK 5 - SAVE
WEIGHTS_PATH = "saans_xray_densenet121.h5"

model.save(WEIGHTS_PATH)
model.save_weights("saans_xray_densenet121.weights.h5")
print(f"saved {WEIGHTS_PATH}  ({os.path.getsize(WEIGHTS_PATH) / 1e6:.1f} MB)")

reloaded = tf.keras.models.load_model(WEIGHTS_PATH)
drift = float(np.max(np.abs(reloaded.predict(test_ds, verbose=0).ravel() - test_prob)))
print(f"reload check - max probability drift: {drift:.2e}")
assert drift < 1e-3, "the reloaded model does not match the trained one"
print("reload OK.")

metadata = {
    "model": "densenet121",
    "version": 2,
    "framework": f"tensorflow-{tf.__version__}",
    "input_shape": [IMG_SIZE, IMG_SIZE, 3],
    "preprocessing": "tf.keras.applications.densenet.preprocess_input (torch mode)",
    "output": "single sigmoid unit, P(TB)",
    "classes": {"0": "Normal", "1": "TB"},
    "threshold": round(float(THRESHOLD), 4),
    "threshold_chosen_for": f"recall >= {TARGET_RECALL} on the validation split",
    "gradcam_layer": "conv5_block16_concat",
    "train_datasets": sorted(df["source"].unique().tolist()),
    "n_train": int(len(y_train)),
    "n_val": int(len(y_val)),
    "n_test": int(len(y_test)),
    "test_metrics": {
        "roc_auc": round(float(auc), 4),
        "pr_auc": round(float(pr_auc), 4),
        "recall": round(float(rec), 4),
        "precision": round(float(prec), 4),
        "accuracy": round(float(acc), 4),
        "specificity": round(float(tn / (tn + fp)), 4),
        "confusion_matrix": cm.tolist(),
    },
    "scope_note": (
        "Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning "
        "is the Phase-1 pilot roadmap item. Chest X-ray is a triage aid, not a diagnosis - "
        "bacteriological confirmation remains the diagnostic standard."
    ),
}
with open("saans_xray_densenet121.json", "w") as fh:
    json.dump(metadata, fh, indent=2)
print(json.dumps(metadata, indent=2))

## BLOCK 6 - Grad-CAM

Unchanged from v1: gradients of the TB **logit** with respect to
`conv5_block16_concat`, pooled into weights, ReLU, normalised, upsampled, blended.
The sigmoid is swapped for a linear activation while taping so gradients survive
a confident prediction, and the feature maps are cast to float32 because mixed
precision leaves them in float16.

In [ ]:
# BLOCK 6 - GRAD-CAM
LAST_CONV_LAYER = "conv5_block16_concat"
_GRAD_MODELS = {}


def _grad_model_for(model, last_conv_layer_name):
    key = (id(model), last_conv_layer_name)
    if key not in _GRAD_MODELS:
        _GRAD_MODELS[key] = tf.keras.models.Model(
            model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
        )
    return _GRAD_MODELS[key]


def make_gradcam_heatmap(preprocessed_batch, model=None, last_conv_layer_name=LAST_CONV_LAYER):
    """Real Grad-CAM. Input (1,224,224,3) already preprocessed. Returns a 7x7 map in [0,1]."""
    model = model if model is not None else globals()["model"]
    head = model.get_layer("tb_output")
    saved_activation = head.activation
    head.activation = tf.keras.activations.linear      # tape the logit, not the squashed probability
    try:
        grad_model = _grad_model_for(model, last_conv_layer_name)
        with tf.GradientTape() as tape:
            conv_out, logit = grad_model(preprocessed_batch, training=False)
            score = logit[:, 0]
        grads = tape.gradient(score, conv_out)
        if grads is None:
            raise RuntimeError("no gradient reached the target conv layer")
        conv_out = tf.cast(conv_out, tf.float32)        # mixed precision leaves these in float16
        grads = tf.cast(grads, tf.float32)
        weights = tf.reduce_mean(grads, axis=(1, 2))
        cam = tf.nn.relu(tf.einsum("bhwc,bc->bhw", conv_out, weights)[0])
        cam = cam / (tf.reduce_max(cam) + 1e-8)
    finally:
        head.activation = saved_activation
    return cam.numpy()


def overlay_heatmap(pil_image, cam, alpha=0.45, size=(512, 512)):
    """Blend a [0,1] Grad-CAM map over the film with a jet colour ramp."""
    base_arr = np.asarray(pil_image.convert("RGB").resize(size, Image.BICUBIC)).astype("float32") / 255.0
    cam_up = np.asarray(
        Image.fromarray(np.uint8(np.clip(cam, 0, 1) * 255)).resize(size, Image.BICUBIC)
    ).astype("float32") / 255.0
    heat = plt.get_cmap("jet")(cam_up)[..., :3]
    mask = (cam_up ** 1.5)[..., None] * alpha
    return Image.fromarray(np.uint8(np.clip(base_arr * (1.0 - mask) + heat * mask, 0, 1) * 255))


def load_for_model(image):
    """Path / PIL image -> (preprocessed (1,224,224,3) batch, original PIL image)."""
    if isinstance(image, (str, os.PathLike, Path)):
        image = Image.open(image)
    image.load()
    arr = np.asarray(image.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype="float32")[None, ...]
    return preprocess_input(arr.copy()), image


print("Grad-CAM target layer:", model.get_layer(LAST_CONV_LAYER).output.shape)

In [ ]:
# BLOCK 6 (cont.) - Grad-CAM on real test films the model got right.
picks = list(df_test.index[(y_true == 1) & (y_pred == 1)][:2]) + list(
    df_test.index[(y_true == 0) & (y_pred == 0)][:1]
)
if not picks:
    picks = list(df_test.index[:3])

fig, axes = plt.subplots(len(picks), 2, figsize=(9, 4.5 * len(picks)))
axes = np.atleast_2d(axes)
for row, i in enumerate(picks):
    path = df_test.loc[i, "path"]
    batch_in, original = load_for_model(path)
    cam = make_gradcam_heatmap(batch_in)
    axes[row, 0].imshow(original.convert("L"), cmap="gray")
    axes[row, 0].set_title(f"{Path(path).name}\ntrue = {CLASS_NAMES[y_true[i]]} · {df_test.loc[i, 'source']}")
    axes[row, 1].imshow(overlay_heatmap(original, cam))
    axes[row, 1].set_title(f"Grad-CAM - P(TB) = {test_prob[i]:.3f}")
    for ax in axes[row]:
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Check the heat sits on lung tissue. If it lands on a corner, a text marker or")
print("the image border, the model has learned something about how the films were")
print("taken rather than about disease - and the score is not trustworthy.")

## BLOCK 7 - Inference

`predict_xray(image)` -> `{"tb_probability": float, "heatmap_image": PIL.Image}`,
plus `heatmap_to_data_url()` for the FastAPI layer. The final cell pushes the
whole test split back through the public function to confirm it reproduces the
Block 4 numbers - the check that catches training/serving preprocessing drift.

In [ ]:
# BLOCK 7 - INFERENCE
# Trained on adult TB CXR datasets as proof-of-concept; paediatric fine-tuning is
# the Phase-1 pilot roadmap item. Every output is decision support for a health
# worker, never a diagnosis.
import base64
import io


def predict_xray(image, model=None, threshold=None, alpha=0.45):
    """
    New chest X-ray in -> TB probability + Grad-CAM overlay out.

    image: file path, PIL.Image, or raw image bytes.
    returns: {"tb_probability": float, "heatmap_image": PIL.Image}
    """
    model = model if model is not None else globals()["model"]
    if isinstance(image, (bytes, bytearray)):
        image = Image.open(io.BytesIO(image))
    batch_in, original = load_for_model(image)
    probability = float(model.predict(batch_in, verbose=0)[0][0])
    cam = make_gradcam_heatmap(batch_in, model)
    return {
        "tb_probability": round(probability, 4),
        "heatmap_image": overlay_heatmap(original, cam, alpha=alpha),
    }


def heatmap_to_data_url(pil_image):
    """PNG data URL, ready for the Saans API response / an <img src>."""
    buf = io.BytesIO()
    pil_image.save(buf, format="PNG", optimize=True)
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


sample_path = df_test.loc[picks[0], "path"]
result = predict_xray(sample_path)
print("input :", Path(sample_path).name, f"(true: {CLASS_NAMES[int(df_test.loc[picks[0], 'label'])]})")
print("output:", {"tb_probability": result["tb_probability"], "heatmap_image": result["heatmap_image"]})
print("verdict:", "TB-suspect" if result["tb_probability"] >= THRESHOLD else "no TB features detected")

plt.figure(figsize=(5, 5))
plt.imshow(result["heatmap_image"])
plt.axis("off")
plt.title(f"predict_xray -> P(TB) = {result['tb_probability']:.3f}")
plt.show()

In [ ]:
# BLOCK 7 (cont.) - sweep a sample of the test split through the public function,
# so what ships is what was measured.
sample = df_test.sample(n=min(300, len(df_test)), random_state=SEED)
sweep = np.array([predict_xray(p)["tb_probability"] for p in tqdm(sample["path"], desc="predict_xray")])
sweep_true = sample["label"].to_numpy().astype(int)
print(f"recall through predict_xray(): {recall_score(sweep_true, (sweep >= THRESHOLD).astype(int)):.4f}")
print(f"reported test recall:          {rec:.4f}")
print("(a sample, so small differences are expected - a large gap means the")
print(" serving path preprocesses differently from training)")

## BLOCK 8 - Download the weights

Run the cell, then move the `.h5` into the repo at
`Saans/backend/models/saans_xray_densenet121.h5`, with the `.json` beside it.

If the download is blocked, use the **folder icon** in Colab's left sidebar,
find the file under `/content`, click the three dots and choose **Download**.

**Backend note:** `server/vision.py` currently loads PyTorch weights. This is a
Keras `.h5`, so `load_model()` / `run_model()` there need swapping to
`tf.keras.models.load_model(...)` plus the Grad-CAM from Block 6 - and the
threshold in the metadata json should drive the decision, not a hardcoded 0.5.

In [ ]:
# BLOCK 8 - download to your machine
from google.colab import files

for name in [
    "saans_xray_densenet121.h5",
    "saans_xray_densenet121.json",
    "training_curves.png",
    "confusion_matrix.png",
]:
    if os.path.exists(name):
        print(f"downloading {name} ({os.path.getsize(name) / 1e6:.1f} MB)")
        files.download(name)
    else:
        print(f"missing: {name} - run the earlier blocks first")